# 01a — Auto-Caption Reference Images

Generates `.txt` sidecar caption files for each reference image using JoyCaption.
These captions are used for SDXL LoRA training in `01b_train_sdxl_lora.ipynb`.

**Runtime:** T4 is fine (captioning is light). A100 optional.

**Flow:**
1. Upload reference images to Drive (or Colab tmp)
2. Run JoyCaption on each image
3. Prepend trigger token to each caption
4. Save `.txt` sidecar files next to each image

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/ai_character_studio'
CHARACTER_NAME = 'Aria'       # ← change this
TRIGGER_TOKEN  = 'ohwx_aria'  # ← change this (must match what you use in training)

import os
CHAR_DIR    = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}'
REF_DIR     = f'{CHAR_DIR}/reference-images'
CAPTION_DIR = f'{CHAR_DIR}/captions'
os.makedirs(REF_DIR, exist_ok=True)
os.makedirs(CAPTION_DIR, exist_ok=True)

print(f'Character: {CHARACTER_NAME} / trigger: {TRIGGER_TOKEN}')
print(f'Reference images dir: {REF_DIR}')
print('Upload your reference images to the Drive folder above, then run the next cells.')

In [ ]:
# (Optional) Upload images directly from this notebook
from google.colab import files
import shutil

print('Select your reference images to upload...')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = f'{REF_DIR}/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved {fname} → {dest}')

In [ ]:
# Install JoyCaption dependencies
!pip install -q transformers torch Pillow huggingface_hub

# Load JoyCaption (fancyfeast/llama-joycaption-alpha-two-vqa-hf)
from transformers import AutoProcessor, LlavaForConditionalGeneration
import torch

MODEL_ID = 'fancyfeast/llama-joycaption-alpha-two-vqa-hf'
print(f'Loading JoyCaption: {MODEL_ID} ...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16
).to('cuda' if torch.cuda.is_available() else 'cpu')
print('JoyCaption loaded.')

In [ ]:
from PIL import Image
from pathlib import Path

CAPTION_INSTRUCTION = (
    'Describe this image in detail for use as a training caption for a diffusion model. '
    'Focus on: the pose and body position, facial expression, clothing and accessories, '
    'lighting and background. Do NOT describe the character identity or face structure — '
    'that will be represented by a trigger token. Be specific and concrete. '
    'Keep the caption under 80 words.'
)

def caption_image(image_path: str) -> str:
    image = Image.open(image_path).convert('RGB')
    conversation = [{
        'role': 'user',
        'content': [{'type': 'image'}, {'type': 'text', 'text': CAPTION_INSTRUCTION}]
    }]
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image], return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generated = processor.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated.strip()

exts = {'.jpg', '.jpeg', '.png', '.webp'}
images = [p for p in Path(REF_DIR).iterdir() if p.suffix.lower() in exts]
print(f'Found {len(images)} reference images.')

captions = {}
for img_path in sorted(images):
    print(f'Captioning {img_path.name} ...')
    raw_caption = caption_image(str(img_path))
    full_caption = f'{TRIGGER_TOKEN}, {raw_caption}'
    captions[img_path.name] = full_caption

    # Save sidecar .txt next to image
    txt_path = Path(REF_DIR) / (img_path.stem + '.txt')
    txt_path.write_text(full_caption)
    # Also save to captions/ folder
    cap_path = Path(CAPTION_DIR) / (img_path.stem + '.txt')
    cap_path.write_text(full_caption)

    print(f'  → {full_caption[:100]}...')

print(f'\nAll {len(captions)} captions saved.')

In [ ]:
# Review and optionally edit captions before training
print('=== Caption Review ===')
for fname, caption in captions.items():
    print(f'\n[{fname}]')
    print(caption)
print('\nEdit the .txt files in Drive if any captions need adjustment before training.')

In [ ]:
# Register character in library (optional — also done in training notebook)
import sys, json
# If running from Colab, library.py isn't installed — use inline version
metadata = {
    'name': CHARACTER_NAME,
    'trigger': TRIGGER_TOKEN,
    'base_model': 'sdxl',
    'ref_count': len(captions),
}
meta_path = f'{CHAR_DIR}/metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'metadata.json written: {meta_path}')
print('\n✅ Done. Run 01b_train_sdxl_lora.ipynb next to train the character LoRA.')